# Packet P-031 — Prediction Monotonicity Audit

**Decision question:** Do the model's predictions respect known physical monotonicity constraints — e.g., does increasing Voc or FF within a composition family consistently increase predicted stability?

**Fastest falsifier:** If any of the top-3 consensus features (bandgap, Jsc, cell_area) show >3 direction reversals in the partial-dependence curve for the majority family (Pure MA), the model is fitting noise rather than physics.

**Success:** Top-3 features monotonic (≤1 reversal) in ≥4/6 families.
**Kill:** Majority of PD curves are non-monotonic with no physical justification.

In [1]:
"""Cell 1 — Load data, train model, classify families."""
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from scipy.stats import kendalltau
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('perovskite_stability_clean.csv')
FEATURES = [
    'Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
    'MA', 'FA', 'Cs',
    'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
    'Perovskite_thickness', 'Cell_area_measured',
    'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF'
]
TARGET = 'Stability_PCE_T80'
ET_PARAMS = dict(n_estimators=700, max_features='sqrt', min_samples_split=3,
                 min_samples_leaf=1, bootstrap=False, random_state=42)

# Top-6 consensus features from P-020
TOP_FEATURES = ['Perovskite_band_gap', 'JV_default_Jsc', 'Cell_area_measured',
                'JV_default_Voc', 'JV_default_FF', 'Perovskite_thickness']

X = df[FEATURES].fillna(0)
y = np.log1p(df[TARGET])

# Train full model
model = ExtraTreesRegressor(**ET_PARAMS)
model.fit(X, y)
print(f"Model trained on {len(X)} devices, {len(FEATURES)} features")

# Classify families
def classify_family(row):
    ma = row["MA"] > 0
    fa = row["FA"] > 0
    cs = row["Cs"] > 0
    if ma and not fa and not cs:
        return "Pure MA"
    elif fa and not ma and not cs:
        return "Pure FA"
    elif ma and fa and not cs:
        return "MA-FA mixed"
    elif fa and cs and not ma:
        return "FA-Cs"
    elif ma and fa and cs:
        return "Triple cation"
    else:
        return "Other"

df["family"] = df.apply(classify_family, axis=1)
family_counts = df["family"].value_counts()
print("\n── Composition family counts ──")
print(family_counts.to_string())

Model trained on 1543 devices, 16 features

── Composition family counts ──
family
Pure MA          967
Triple cation    197
MA-FA mixed      197
Other             88
Pure FA           50
FA-Cs             44


In [2]:
"""Cell 2 — Compute partial-dependence curves per feature per family."""

# For each top feature, sweep it across its observed range while holding
# all other features at the family-specific median.

N_GRID = 50  # grid points per feature

results = []  # (feature, family, n_reversals, direction, grid_values, pd_values)

families = [f for f in family_counts.index if family_counts[f] >= 30]
print(f"Testing {len(TOP_FEATURES)} features x {len(families)} families\n")

for feat in TOP_FEATURES:
    feat_idx = FEATURES.index(feat)
    for fam in families:
        mask = df["family"] == fam
        X_fam = X[mask]
        
        # Median baseline for this family
        baseline = X_fam.median().values.copy()
        
        # Sweep range: 5th to 95th percentile of this feature in this family
        lo = X_fam[feat].quantile(0.05)
        hi = X_fam[feat].quantile(0.95)
        
        # Skip if no variation
        if hi - lo < 1e-8:
            results.append({
                'feature': feat, 'family': fam, 'n_reversals': 0,
                'direction': 'constant', 'range': f"{lo:.3f}-{hi:.3f}",
                'note': 'No variation in family'
            })
            continue
        
        grid = np.linspace(lo, hi, N_GRID)
        
        # Build synthetic dataset: baseline repeated, only target feature varies
        X_synth = np.tile(baseline, (N_GRID, 1))
        X_synth[:, feat_idx] = grid
        
        # Predict
        preds = model.predict(X_synth)
        
        # Count monotonicity reversals
        diffs = np.diff(preds)
        signs = np.sign(diffs)
        signs_nonzero = signs[signs != 0]
        if len(signs_nonzero) == 0:
            n_reversals = 0
            direction = 'flat'
        else:
            reversals = np.sum(np.diff(signs_nonzero) != 0)
            n_reversals = int(reversals)
            # Overall direction: positive if more positive diffs
            direction = 'increasing' if np.sum(diffs > 0) > np.sum(diffs < 0) else 'decreasing'
        
        results.append({
            'feature': feat, 'family': fam, 'n_reversals': n_reversals,
            'direction': direction, 'range': f"{lo:.3f}-{hi:.3f}",
            'note': ''
        })

res_df = pd.DataFrame(results)
print("── Partial Dependence Monotonicity Summary ──\n")
for feat in TOP_FEATURES:
    sub = res_df[res_df['feature'] == feat]
    print(f"\n{feat}:")
    for _, row in sub.iterrows():
        flag = " ⚠" if row['n_reversals'] > 3 else " ✓"
        print(f"  {row['family']:<20} reversals={row['n_reversals']:>2}  dir={row['direction']:<12} range={row['range']}{flag}")

Testing 6 features x 6 families



── Partial Dependence Monotonicity Summary ──


Perovskite_band_gap:
  Pure MA              reversals= 7  dir=decreasing   range=0.000-1.610 ⚠
  Triple cation        reversals=15  dir=increasing   range=0.000-1.620 ⚠
  MA-FA mixed          reversals=23  dir=decreasing   range=0.000-1.600 ⚠
  Other                reversals=21  dir=increasing   range=0.000-2.008 ⚠
  Pure FA              reversals= 0  dir=decreasing   range=0.000-1.491 ✓
  FA-Cs                reversals=21  dir=increasing   range=0.000-1.670 ⚠

JV_default_Jsc:
  Pure MA              reversals=10  dir=increasing   range=9.565-23.467 ⚠
  Triple cation        reversals=16  dir=increasing   range=12.266-24.080 ⚠
  MA-FA mixed          reversals= 5  dir=increasing   range=0.000-24.212 ⚠
  Other                reversals= 9  dir=increasing   range=0.144-19.228 ⚠
  Pure FA              reversals=20  dir=increasing   range=12.488-24.441 ⚠
  FA-Cs                reversals=21  dir=increasing   range=16.578-24.880 ⚠

Cell_area_measur

In [3]:
"""Cell 3 — Evaluate against thresholds and save."""

# Success: top-3 features monotonic (<=1 reversal) in >=4/6 families
# Kill: majority of PD curves non-monotonic

TOP3 = TOP_FEATURES[:3]  # bandgap, Jsc, cell_area

# Count families where each top-3 feature passes (<=1 reversal)
top3_pass = {}
for feat in TOP3:
    sub = res_df[(res_df['feature'] == feat) & (res_df['note'] != 'No variation in family')]
    n_pass = (sub['n_reversals'] <= 1).sum()
    n_total = len(sub)
    top3_pass[feat] = (n_pass, n_total)
    print(f"{feat}: {n_pass}/{n_total} families pass (<=1 reversal)")

# Overall stats
total_curves = len(res_df[res_df['note'] != 'No variation in family'])
monotonic_curves = len(res_df[(res_df['n_reversals'] <= 1) & (res_df['note'] != 'No variation in family')])
high_reversal = len(res_df[res_df['n_reversals'] > 3])

print(f"\nOverall: {monotonic_curves}/{total_curves} curves monotonic (<=1 reversal)")
print(f"High-reversal (>3): {high_reversal}/{total_curves} curves")

# Check Pure MA specifically (fastest falsifier)
pure_ma = res_df[(res_df['family'] == 'Pure MA') & (res_df['feature'].isin(TOP3))]
ma_fail = pure_ma[pure_ma['n_reversals'] > 3]
if len(ma_fail) > 0:
    print(f"\n⚠ FASTEST FALSIFIER: {len(ma_fail)} top-3 features have >3 reversals in Pure MA:")
    for _, r in ma_fail.iterrows():
        print(f"  {r['feature']}: {r['n_reversals']} reversals")
else:
    print(f"\n✓ Fastest falsifier passed: all top-3 features have <=3 reversals in Pure MA")

# Determine status
# Success: top-3 features pass in >=4 families each
top3_success = all(p[0] >= 4 for p in top3_pass.values())
# Kill: majority (>50%) of curves are non-monotonic
majority_nonmonotonic = monotonic_curves < total_curves * 0.5

if majority_nonmonotonic:
    status = "Negative"
elif top3_success:
    status = "Confirmed"
elif monotonic_curves >= total_curves * 0.5:
    status = "Promising"
else:
    status = "Negative"

# Save
res_df.to_csv('Packet_P031_Prediction_Monotonicity.csv', index=False)
print(f"\nSaved: Packet_P031_Prediction_Monotonicity.csv")

print(f"\n{'=' * 60}")
print(f"P-031 STATUS: {status}")
print(f"{'=' * 60}")
print(f"Monotonic curves: {monotonic_curves}/{total_curves}")
print(f"Top-3 pass rates: {', '.join(f'{k}: {v[0]}/{v[1]}' for k, v in top3_pass.items())}")
if status == "Confirmed":
    print("PD curves are broadly monotonic — model captures physical trends.")
elif status == "Promising":
    print("Mixed monotonicity — some features respect physics, others don't.")
else:
    print("Model predictions do not respect physical monotonicity constraints.")

Perovskite_band_gap: 1/6 families pass (<=1 reversal)
JV_default_Jsc: 0/6 families pass (<=1 reversal)
Cell_area_measured: 0/6 families pass (<=1 reversal)

Overall: 1/36 curves monotonic (<=1 reversal)
High-reversal (>3): 34/36 curves

⚠ FASTEST FALSIFIER: 3 top-3 features have >3 reversals in Pure MA:
  Perovskite_band_gap: 7 reversals
  JV_default_Jsc: 10 reversals
  Cell_area_measured: 11 reversals

Saved: Packet_P031_Prediction_Monotonicity.csv

P-031 STATUS: Negative
Monotonic curves: 1/36
Top-3 pass rates: Perovskite_band_gap: 1/6, JV_default_Jsc: 0/6, Cell_area_measured: 0/6
Model predictions do not respect physical monotonicity constraints.
